# Qwen3.5-2B LoRA on ViOCRVQA + ViTextVQA

Notebook này fine-tune `Qwen/Qwen3.5-2B` bằng LoRA trên **cả 2 tập** `ViOCRVQA` và `ViTextVQA`, tối ưu cho **1 GPU H100 trên `cuda:0`**.

Mặc định notebook dùng:
- `bf16` + LoRA, không dùng QLoRA vì H100 dư VRAM và LoRA bf16 nhanh hơn, ổn định hơn.
- temperature-mixing giữa 2 tập với `alpha=0.7` để tránh tập lớn lấn át tập nhỏ.
- lưu checkpoint theo epoch, rồi **chọn checkpoint tốt nhất theo `ANLS` trên dev** thay vì chỉ theo `eval_loss`.
- báo cáo cả metric **gộp 2 dev set** và **theo từng dataset**.

Trước khi chạy, chỉ cần bảo đảm thư mục dữ liệu đúng như bạn mô tả:

```text
data/
├── ViOCRVQA/
│   ├── bf_all_img/
│   ├── train.json
│   └── dev.json
└── ViTextVQA/
    ├── st_images/
    ├── ViTextVQA_train.json
    └── ViTextVQA_dev.json
```

Nếu bạn muốn đổi sang `Qwen/Qwen3.5-2B-Base`, chỉ cần sửa `MODEL_ID` ở cell config.

In [ ]:
# Chạy 1 lần nếu environment chưa có support Qwen3.5 mới nhất.
# flash-attn là tùy chọn: nếu cài được trên H100 thì để ATTN_IMPLEMENTATION='flash_attention_2',
# còn không notebook sẽ tự fallback sang SDPA.

%pip install -q -U \
  "transformers @ git+https://github.com/huggingface/transformers.git@main" \
  peft accelerate bitsandbytes qwen-vl-utils pillow tqdm python-Levenshtein packaging

# Tùy chọn nếu image extension nhiều và muốn FA2 thật sự:
# %pip install -q -U flash-attn --no-build-isolation

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = REPO_ROOT / 'data'
OUTPUT_ROOT = REPO_ROOT / 'checkpoints' / 'qwen35_2b_lora_multidataset'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Qwen/Qwen3.5-2B'
DEVICE = 'cuda:0'
SEED = 42

TRAIN_SOURCES = {
    'ViTextVQA': {
        'train_json': DATA_ROOT / 'ViTextVQA' / 'ViTextVQA_train.json',
        'dev_json': DATA_ROOT / 'ViTextVQA' / 'ViTextVQA_dev.json',
        'image_dir': DATA_ROOT / 'ViTextVQA' / 'st_images',
    },
    'ViOCRVQA': {
        'train_json': DATA_ROOT / 'ViOCRVQA' / 'train.json',
        'dev_json': DATA_ROOT / 'ViOCRVQA' / 'dev.json',
        'image_dir': DATA_ROOT / 'ViOCRVQA' / 'bf_all_img',
    },
}

# Sampling / training strategy
MIXING_ALPHA = 0.7          # 0.0 = 50/50 tuyệt đối, 1.0 = tỉ lệ theo kích thước tập
TOTAL_TRAIN_SAMPLES = None  # None = tổng số sample train của cả 2 tập
NUM_EPOCHS = 2
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4   # effective batch = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
LR_SCHEDULER_TYPE = 'cosine'
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 3
DATALOADER_NUM_WORKERS = 4

# Vision resolution: OCR cần ảnh khá nét; 1024 là điểm cân bằng tốt trên H100.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28
ATTN_IMPLEMENTATION = 'flash_attention_2'  # notebook sẽ fallback sang 'sdpa' nếu thiếu flash-attn
TRUST_REMOTE_CODE = True
ENABLE_THINKING = False

# LoRA config
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_MODULES = 'all-linear'  # an toàn cho Qwen3.5 nếu chưa muốn hard-code tên module
BIAS = 'none'

# Prompt / generation
MAX_NEW_TOKENS = 32
NUM_BEAMS = 1
DO_SAMPLE_EVAL = False
SYSTEM_PROMPT = (
    'Bạn là trợ lý VQA tiếng Việt. Trả lời chỉ dựa trên nội dung nhìn thấy trong ảnh, '
    'ưu tiên ngắn gọn và không bịa thêm thông tin.'
)
USER_PROMPT_TEMPLATE = (
    'Hãy trả lời ngắn gọn chỉ bằng đáp án cuối cùng, không giải thích thêm.\\n'
    'Câu hỏi: {question}'
)

IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
SKIP_MISSING_IMAGES = False
DEV_EVAL_LIMIT = None

CONFIG = {
    'repo_root': str(REPO_ROOT),
    'data_root': str(DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'model_id': MODEL_ID,
    'device': DEVICE,
    'mixing_alpha': MIXING_ALPHA,
    'epochs': NUM_EPOCHS,
    'train_batch_size': PER_DEVICE_TRAIN_BATCH_SIZE,
    'eval_batch_size': PER_DEVICE_EVAL_BATCH_SIZE,
    'grad_accum': GRADIENT_ACCUMULATION_STEPS,
    'lr': LEARNING_RATE,
    'min_pixels': MIN_PIXELS,
    'max_pixels': MAX_PIXELS,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
}
CONFIG

In [ ]:
import json
import math
import os
import random
import re
import unicodedata
from collections import Counter
from functools import lru_cache
from importlib import import_module
from importlib.metadata import PackageNotFoundError, version as package_version

import numpy as np
import torch
from packaging import version
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    current_transformers_version = package_version('transformers')
except PackageNotFoundError as exc:
    raise RuntimeError('Thiếu transformers. Hãy chạy cell cài đặt ở trên.') from exc

if version.parse(current_transformers_version) < version.parse('5.2.0'):
    raise RuntimeError(
        f'transformers={current_transformers_version} quá cũ cho Qwen3.5. '
        "Hãy cài từ `transformers@main` như cell phía trên."
    )

transformers_module = import_module('transformers')
AutoProcessor = getattr(transformers_module, 'AutoProcessor')
Trainer = getattr(transformers_module, 'Trainer')
TrainingArguments = getattr(transformers_module, 'TrainingArguments')


def resolve_qwen35_model_class():
    for class_name in ('Qwen3_5ForConditionalGeneration', 'AutoModelForImageTextToText', 'AutoModelForVision2Seq'):
        model_class = getattr(transformers_module, class_name, None)
        if model_class is not None:
            return class_name, model_class
    raise ImportError('Không tìm thấy class load model cho Qwen3.5 trong transformers hiện tại.')


MODEL_CLASS_NAME, Qwen35ModelClass = resolve_qwen35_model_class()

from peft import LoraConfig, PeftModel, TaskType, get_peft_model


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

assert torch.cuda.is_available(), 'Notebook này đang cấu hình cho H100 / CUDA nhưng không thấy GPU.'
assert str(torch.device(DEVICE)) == 'cuda:0', f'DEVICE hiện tại là {DEVICE}, notebook mong đợi cuda:0.'

for source_name, spec in TRAIN_SOURCES.items():
    for key in ('train_json', 'dev_json', 'image_dir'):
        path = spec[key]
        if not path.exists():
            raise FileNotFoundError(f'[{source_name}] Không thấy: {path}')

print({
    'seed': SEED,
    'transformers_version': current_transformers_version,
    'model_class': MODEL_CLASS_NAME,
    'device': DEVICE,
    'output_root': str(OUTPUT_ROOT),
})

In [ ]:
ZERO_WIDTH_RE = re.compile(r'[\u200b\u200c\u200d\ufeff]')
EDGE_PUNCT_RE = re.compile(r"^[\s\.,!?;:'\"“”‘’`~_]+|[\s\.,!?;:'\"“”‘’`~_]+$")
DASH_TRANSLATION = str.maketrans({'–': '-', '—': '-', '−': '-', '‐': '-'})
SIMPLE_NUMBER_WORDS = {
    'không': '0', 'một': '1', 'hai': '2', 'ba': '3', 'bốn': '4', 'tư': '4',
    'năm': '5', 'lăm': '5', 'sáu': '6', 'bảy': '7', 'bẩy': '7', 'tám': '8',
    'chín': '9', 'mười': '10', 'zero': '0', 'one': '1', 'two': '2', 'three': '3',
    'four': '4', 'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9', 'ten': '10',
}


def _canonicalize_number(text):
    compact = text.replace(' ', '')
    if re.fullmatch(r'\d{1,3}([.,]\d{3})+', compact):
        return re.sub(r'[.,]', '', compact)
    if re.fullmatch(r'\d+,\d+', compact) and not re.fullmatch(r'\d{1,3}(,\d{3})+', compact):
        return compact.replace(',', '.')
    return text


def normalize_answer(text):
    if text is None:
        return ''
    text = str(text)
    text = unicodedata.normalize('NFC', text)
    text = text.translate(DASH_TRANSLATION)
    text = ZERO_WIDTH_RE.sub('', text)
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = EDGE_PUNCT_RE.sub('', text)
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF\-\./,]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if text in SIMPLE_NUMBER_WORDS:
        text = SIMPLE_NUMBER_WORDS[text]
    text = _canonicalize_number(text)
    text = re.sub(r'^[\.,!?;:]+|[\.,!?;:]+$', '', text).strip()
    return re.sub(r'\s+', ' ', text)


def anls_score(prediction, ground_truths, threshold=0.5):
    from Levenshtein import distance as levenshtein_distance

    pred = normalize_answer(prediction)
    max_sim = 0.0
    for gt in ground_truths:
        gt = normalize_answer(gt)
        nl = max(len(pred), len(gt))
        if nl == 0:
            sim = 1.0
        else:
            sim = 1.0 - levenshtein_distance(pred, gt) / nl
        max_sim = max(max_sim, sim)
    return max_sim if max_sim >= threshold else 0.0


def exact_match_score(prediction, ground_truths):
    pred = normalize_answer(prediction)
    return float(any(normalize_answer(gt) == pred for gt in ground_truths))


def compute_metrics(predictions, ground_truths):
    assert len(predictions) == len(ground_truths)
    if not predictions:
        return {'anls': 0.0, 'exact_match': 0.0, 'num_samples': 0}
    total_anls = sum(anls_score(pred, gts) for pred, gts in zip(predictions, ground_truths))
    total_em = sum(exact_match_score(pred, gts) for pred, gts in zip(predictions, ground_truths))
    n = len(predictions)
    return {
        'anls': total_anls / n,
        'exact_match': total_em / n,
        'num_samples': n,
    }


def load_annotations(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ('annotations', 'data', 'items', 'samples', 'questions'):
            if key in data and isinstance(data[key], list):
                return data[key]
    raise ValueError(f'Unsupported annotation format: {json_path}')


def _extract_scalar(value):
    if value is None:
        return None
    if isinstance(value, dict):
        for key in ('answer', 'text', 'label', 'value'):
            if value.get(key) is not None:
                return str(value[key])
        return json.dumps(value, ensure_ascii=False)
    return str(value)


def extract_question(ann):
    for key in ('question', 'query', 'text'):
        value = ann.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    raise KeyError(f'Cannot find question field in sample: {ann}')


def extract_answers(ann):
    if isinstance(ann.get('answers'), list):
        answers = [_extract_scalar(x) for x in ann['answers']]
    elif isinstance(ann.get('all_answers'), list):
        answers = [_extract_scalar(x) for x in ann['all_answers']]
    elif ann.get('answer') is not None:
        answers = [_extract_scalar(ann['answer'])]
    elif ann.get('label') is not None:
        label = ann['label']
        if isinstance(label, list):
            answers = [_extract_scalar(x) for x in label]
        else:
            answers = [_extract_scalar(label)]
    else:
        answers = ['']
    cleaned = [str(x).strip() for x in answers if x is not None and str(x).strip()]
    return cleaned or ['']


def extract_image_identifier(ann):
    for key in ('image_id', 'image', 'image_name', 'file_name', 'filename', 'img_id', 'img'):
        value = ann.get(key)
        if value is not None:
            return str(value)
    raise KeyError(f'Cannot find image identifier field in sample: {ann}')


def extract_question_id(ann, source_name, idx):
    for key in ('id', 'question_id', 'qid'):
        value = ann.get(key)
        if value is not None:
            return str(value)
    return f'{source_name}_{idx}'


@lru_cache(maxsize=200000)
def resolve_image_path(image_identifier, image_dir, image_exts=IMAGE_EXTS):
    image_dir = Path(image_dir)
    candidate = Path(str(image_identifier))
    if candidate.suffix:
        direct = image_dir / candidate.name
        if direct.exists():
            return direct
        if candidate.exists():
            return candidate
    stem = candidate.stem if candidate.suffix else candidate.name
    for ext in image_exts:
        path = image_dir / f'{stem}{ext}'
        if path.exists():
            return path
    matches = sorted(image_dir.glob(f'{stem}.*'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Cannot find image for id '{image_identifier}' in {image_dir}")


def canonicalize_records(source_name, split_name, annotation_path, image_dir):
    raw_items = load_annotations(annotation_path)
    records = []
    missing = []
    for idx, ann in enumerate(raw_items):
        image_id = extract_image_identifier(ann)
        try:
            image_path = resolve_image_path(image_id, image_dir)
        except FileNotFoundError:
            if SKIP_MISSING_IMAGES:
                missing.append(image_id)
                continue
            raise
        answers = extract_answers(ann)
        records.append({
            'source': source_name,
            'split': split_name,
            'question_id': extract_question_id(ann, source_name, idx),
            'image_id': image_id,
            'image_path': str(image_path),
            'question': extract_question(ann),
            'answers': answers,
            'answer': answers[0],
        })
    return records, missing


def build_mixed_records(records_by_source, alpha=0.7, total_samples=None, seed=42):
    source_names = list(records_by_source.keys())
    lengths = np.array([len(records_by_source[name]) for name in source_names], dtype=np.int64)
    assert np.all(lengths > 0), 'Tất cả source phải có ít nhất 1 sample train.'
    total_samples = int(total_samples or lengths.sum())
    weights = lengths.astype(np.float64) ** alpha
    probs = weights / weights.sum()
    raw_counts = probs * total_samples
    counts = np.floor(raw_counts).astype(int)
    remainder = total_samples - counts.sum()
    if remainder > 0:
        order = np.argsort(-(raw_counts - counts))
        for idx in order[:remainder]:
            counts[idx] += 1

    rng = np.random.default_rng(seed)
    mixed_records = []
    mix_plan = []
    for source_idx, source_name in enumerate(source_names):
        source_records = records_by_source[source_name]
        count = int(counts[source_idx])
        replace = count > len(source_records)
        sampled_indices = rng.choice(len(source_records), size=count, replace=replace)
        for sample_idx in sampled_indices:
            mixed_records.append(dict(source_records[int(sample_idx)]))
        mix_plan.append({
            'source': source_name,
            'original_size': int(len(source_records)),
            'sampled_size': count,
            'probability': float(probs[source_idx]),
            'replace': bool(replace),
        })
    rng.shuffle(mixed_records)
    return mixed_records, mix_plan


class UnifiedVQADataset(Dataset):
    def __init__(self, records):
        self.records = list(records)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = dict(self.records[idx])
        record['image'] = Image.open(record['image_path']).convert('RGB')
        return record


print(normalize_answer('  Một—nghìn  '))
print(normalize_answer('1.000'))

In [ ]:
train_records_by_source = {}
dev_records_by_source = {}
missing_images = {}

for source_name, spec in TRAIN_SOURCES.items():
    train_records, missing_train = canonicalize_records(
        source_name=source_name,
        split_name='train',
        annotation_path=spec['train_json'],
        image_dir=spec['image_dir'],
    )
    dev_records, missing_dev = canonicalize_records(
        source_name=source_name,
        split_name='dev',
        annotation_path=spec['dev_json'],
        image_dir=spec['image_dir'],
    )
    train_records_by_source[source_name] = train_records
    dev_records_by_source[source_name] = dev_records
    missing_images[source_name] = {
        'train_missing': len(missing_train),
        'dev_missing': len(missing_dev),
    }

mixed_train_records, mix_plan = build_mixed_records(
    train_records_by_source,
    alpha=MIXING_ALPHA,
    total_samples=TOTAL_TRAIN_SAMPLES,
    seed=SEED,
)

merged_dev_records = []
for source_name in TRAIN_SOURCES:
    merged_dev_records.extend(dev_records_by_source[source_name])

if DEV_EVAL_LIMIT is not None:
    merged_dev_records = merged_dev_records[:DEV_EVAL_LIMIT]

train_dataset = UnifiedVQADataset(mixed_train_records)
dev_dataset = UnifiedVQADataset(merged_dev_records)
dev_datasets_by_source = {
    source_name: UnifiedVQADataset(records)
    for source_name, records in dev_records_by_source.items()
}

stats = {
    'train_sizes': {k: len(v) for k, v in train_records_by_source.items()},
    'dev_sizes': {k: len(v) for k, v in dev_records_by_source.items()},
    'mixed_train_size': len(mixed_train_records),
    'merged_dev_size': len(merged_dev_records),
    'mix_plan': mix_plan,
    'missing_images': missing_images,
}
stats

In [ ]:
def resolve_attn_implementation(requested_impl):
    if requested_impl is None:
        return None
    if requested_impl != 'flash_attention_2':
        return requested_impl
    try:
        import flash_attn  # noqa: F401
        return requested_impl
    except Exception as exc:
        print(f'flash-attn không khả dụng ({exc}); fallback sang sdpa.')
        return 'sdpa'


def build_messages(question, image, answer=None):
    messages = []
    if SYSTEM_PROMPT.strip():
        messages.append({
            'role': 'system',
            'content': [{'type': 'text', 'text': SYSTEM_PROMPT.strip()}],
        })
    messages.append({
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': USER_PROMPT_TEMPLATE.format(question=question).strip()},
        ],
    })
    if answer is not None:
        messages.append({
            'role': 'assistant',
            'content': [{'type': 'text', 'text': str(answer).strip()}],
        })
    return messages


def apply_qwen35_chat_template(processor, conversations, add_generation_prompt, padding):
    common_kwargs = {
        'tokenize': True,
        'return_dict': True,
        'return_tensors': 'pt',
        'padding': padding,
        'add_generation_prompt': add_generation_prompt,
    }
    attempts = [
        {'enable_thinking': ENABLE_THINKING},
        {'chat_template_kwargs': {'enable_thinking': ENABLE_THINKING}},
        {},
    ]
    last_error = None
    for extra_kwargs in attempts:
        try:
            return processor.apply_chat_template(conversations, **common_kwargs, **extra_kwargs)
        except TypeError as exc:
            last_error = exc
    raise last_error


def pad_1d(sequences, pad_value, max_len, left=False):
    padded = []
    for seq in sequences:
        pad_len = max_len - seq.shape[0]
        pad = torch.full((pad_len,), pad_value, dtype=seq.dtype)
        padded.append(torch.cat([pad, seq]) if left else torch.cat([seq, pad]))
    return torch.stack(padded)


class Qwen35VQACollator:
    def __init__(self, processor, training=True):
        self.processor = processor
        self.training = training
        tokenizer = getattr(processor, 'tokenizer', None)
        self.pad_token_id = 0
        if tokenizer is not None:
            if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
                tokenizer.pad_token = tokenizer.eos_token
            self.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id or 0

    def _encode(self, messages, add_generation_prompt):
        return apply_qwen35_chat_template(
            self.processor,
            conversations=[messages],
            add_generation_prompt=add_generation_prompt,
            padding=False,
        )

    def __call__(self, batch):
        input_ids_list = []
        attention_masks = []
        labels_list = []
        seq_like_tensors = {}
        vision_like_tensors = {}

        for item in batch:
            prompt_messages = build_messages(item['question'], item['image'])
            if self.training:
                full_messages = build_messages(item['question'], item['image'], item['answer'])
                prompt_enc = self._encode(prompt_messages, add_generation_prompt=True)
                enc = self._encode(full_messages, add_generation_prompt=False)
                labels = enc['input_ids'][0].clone()
                prompt_len = prompt_enc['input_ids'][0].shape[0]
                labels[:prompt_len] = -100
                labels_list.append(labels)
            else:
                enc = self._encode(prompt_messages, add_generation_prompt=True)

            ids = enc['input_ids'][0]
            mask = enc['attention_mask'][0]
            input_ids_list.append(ids)
            attention_masks.append(mask)

            for key, value in enc.items():
                if key in {'input_ids', 'attention_mask'} or not isinstance(value, torch.Tensor):
                    continue
                if value.ndim == 2 and value.shape[0] == 1 and value.shape[1] == ids.shape[0]:
                    seq_like_tensors.setdefault(key, []).append(value[0])
                else:
                    vision_like_tensors.setdefault(key, []).append(value)

        max_len = max(x.shape[0] for x in input_ids_list)
        left_pad = not self.training
        result = {
            'input_ids': pad_1d(input_ids_list, self.pad_token_id, max_len, left=left_pad),
            'attention_mask': pad_1d(attention_masks, 0, max_len, left=left_pad),
        }
        if labels_list:
            result['labels'] = pad_1d(labels_list, -100, max_len, left=False)
        for key, tensors in seq_like_tensors.items():
            result[key] = pad_1d(tensors, 0, max_len, left=left_pad)
        for key, tensors in vision_like_tensors.items():
            result[key] = torch.cat(tensors, dim=0)
        if not self.training:
            result['meta'] = [
                {
                    'question_id': item['question_id'],
                    'image_id': item['image_id'],
                    'question': item['question'],
                    'ground_truths': item['answers'],
                    'source': item['source'],
                }
                for item in batch
            ]
        return result


def move_batch_to_device(batch, device):
    return {
        key: (value.to(device) if isinstance(value, torch.Tensor) else value)
        for key, value in batch.items()
        if key != 'meta'
    }


def build_lora_model():
    attn_impl = resolve_attn_implementation(ATTN_IMPLEMENTATION)
    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        trust_remote_code=TRUST_REMOTE_CODE,
    )
    tokenizer = getattr(processor, 'tokenizer', None)
    if tokenizer is not None and tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    load_kwargs = {
        'torch_dtype': torch_dtype,
        'trust_remote_code': TRUST_REMOTE_CODE,
    }
    if attn_impl is not None:
        load_kwargs['attn_implementation'] = attn_impl

    model = Qwen35ModelClass.from_pretrained(MODEL_ID, **load_kwargs)
    if hasattr(model.config, 'use_cache'):
        model.config.use_cache = False
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
        bias=BIAS,
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, processor, attn_impl, torch_dtype


model, processor, resolved_attn_implementation, torch_dtype = build_lora_model()
train_collator = Qwen35VQACollator(processor, training=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_ROOT),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    logging_steps=LOGGING_STEPS,
    save_strategy='epoch',
    save_total_limit=SAVE_TOTAL_LIMIT,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim='adamw_torch_fused',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=DATALOADER_NUM_WORKERS > 0,
    remove_unused_columns=False,
    report_to='none',
    seed=SEED,
    save_safetensors=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=train_collator,
)

print({
    'resolved_attn_implementation': resolved_attn_implementation,
    'torch_dtype': str(torch_dtype),
    'train_samples': len(train_dataset),
    'dev_samples': len(dev_dataset),
})

In [ ]:
train_result = trainer.train()
trainer.save_model(str(OUTPUT_ROOT / 'final_adapter'))
processor.save_pretrained(OUTPUT_ROOT / 'processor')

with open(OUTPUT_ROOT / 'training_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

train_result.metrics

In [ ]:
def postprocess_prediction(text):
    text = str(text or '')
    text = re.sub(r'<think>.*?</think>', ' ', text, flags=re.S)
    text = text.replace('<|im_end|>', ' ')
    text = text.strip()
    if '\n' in text:
        text = text.splitlines()[0].strip()
    text = re.sub(r'^(trả lời|đáp án|answer)\s*[:\-]\s*', '', text, flags=re.I).strip()
    return text


def load_eval_model(checkpoint_dir):
    attn_impl = resolved_attn_implementation
    load_kwargs = {
        'torch_dtype': torch_dtype,
        'trust_remote_code': TRUST_REMOTE_CODE,
    }
    if attn_impl is not None:
        load_kwargs['attn_implementation'] = attn_impl

    base_model = Qwen35ModelClass.from_pretrained(MODEL_ID, **load_kwargs)
    model = PeftModel.from_pretrained(base_model, str(checkpoint_dir))
    model = model.merge_and_unload()
    if hasattr(model.config, 'use_cache'):
        model.config.use_cache = True
    model = model.to(torch.device(DEVICE))
    model.eval()
    return model


@torch.no_grad()
def evaluate_checkpoint(checkpoint_dir, eval_dataset):
    eval_model = load_eval_model(checkpoint_dir)
    eval_collator = Qwen35VQACollator(processor, training=False)
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=DATALOADER_NUM_WORKERS,
        pin_memory=True,
        persistent_workers=DATALOADER_NUM_WORKERS > 0,
        collate_fn=eval_collator,
    )

    tokenizer = getattr(processor, 'tokenizer', None)
    pad_token_id = 0
    if tokenizer is not None:
        pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id or 0

    predictions = []
    results = []
    for batch in tqdm(eval_loader, desc=f'Evaluating {Path(checkpoint_dir).name}'):
        meta = batch.pop('meta')
        tensor_batch = move_batch_to_device(batch, torch.device(DEVICE))
        generated = eval_model.generate(
            **tensor_batch,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            do_sample=DO_SAMPLE_EVAL,
            pad_token_id=pad_token_id,
        )
        input_len = tensor_batch['input_ids'].shape[1]
        if tokenizer is not None:
            decoded = tokenizer.batch_decode(generated[:, input_len:], skip_special_tokens=True)
        else:
            decoded = processor.batch_decode(generated[:, input_len:], skip_special_tokens=True)

        for item, pred in zip(meta, decoded):
            pred = postprocess_prediction(pred)
            predictions.append(pred)
            results.append({
                'source': item['source'],
                'question_id': item['question_id'],
                'image_id': item['image_id'],
                'question': item['question'],
                'prediction': pred,
                'ground_truths': item['ground_truths'],
            })

    ground_truths = [x['ground_truths'] for x in results]
    overall_metrics = compute_metrics(predictions, ground_truths)
    per_source_metrics = {}
    for source_name in sorted({x['source'] for x in results}):
        source_results = [x for x in results if x['source'] == source_name]
        source_preds = [x['prediction'] for x in source_results]
        source_gts = [x['ground_truths'] for x in source_results]
        per_source_metrics[source_name] = compute_metrics(source_preds, source_gts)

    del eval_model
    torch.cuda.empty_cache()

    return {
        'checkpoint': str(checkpoint_dir),
        'overall': overall_metrics,
        'per_source': per_source_metrics,
        'predictions': results,
    }


checkpoint_dirs = sorted(
    [p for p in OUTPUT_ROOT.glob('checkpoint-*') if p.is_dir()],
    key=lambda p: int(p.name.split('-')[-1]),
)
if (OUTPUT_ROOT / 'final_adapter').exists():
    checkpoint_dirs.append(OUTPUT_ROOT / 'final_adapter')

assert checkpoint_dirs, f'Không tìm thấy checkpoint nào trong {OUTPUT_ROOT}'

all_dev_results = []
for checkpoint_dir in checkpoint_dirs:
    dev_result = evaluate_checkpoint(checkpoint_dir, dev_dataset)
    all_dev_results.append(dev_result)
    print({
        'checkpoint': dev_result['checkpoint'],
        'overall_anls': round(dev_result['overall']['anls'] * 100, 2),
        'overall_em': round(dev_result['overall']['exact_match'] * 100, 2),
        'per_source': {
            k: {
                'anls': round(v['anls'] * 100, 2),
                'em': round(v['exact_match'] * 100, 2),
            }
            for k, v in dev_result['per_source'].items()
        },
    })

all_dev_results = sorted(all_dev_results, key=lambda x: x['overall']['anls'], reverse=True)
best_dev_result = all_dev_results[0]

summary = {
    'model_id': MODEL_ID,
    'best_checkpoint': best_dev_result['checkpoint'],
    'best_overall': best_dev_result['overall'],
    'best_per_source': best_dev_result['per_source'],
    'all_checkpoints': [
        {
            'checkpoint': x['checkpoint'],
            'overall': x['overall'],
            'per_source': x['per_source'],
        }
        for x in all_dev_results
    ],
    'config': CONFIG,
}

with open(OUTPUT_ROOT / 'dev_eval_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
with open(OUTPUT_ROOT / 'best_dev_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(best_dev_result['predictions'], f, ensure_ascii=False, indent=2)

summary